In [1]:
spark_version = "spark-4.0.1"
spark_version

'spark-4.0.1'

In [2]:
spark_release = spark_version
hadoop_version = "hadoop3"

import os, time
start = time.time()

os.environ["SPARK_RELEASE"] = spark_release
os.environ["HADOOP_VERSION"] = hadoop_version
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = f"/content/{spark_release}-bin-{hadoop_version}"

In [3]:
!apt-get update
!apt-get install openjdk-21-jdk-headless -qq > /dev/null

!wget https://archive.apache.org/dist/spark/${SPARK_RELEASE}/${SPARK_RELEASE}-bin-${HADOOP_VERSION}.tgz
!tar xf ${SPARK_RELEASE}-bin-${HADOOP_VERSION}.tgz

!pip install -q findspark

import os
print("SPARK_HOME =", os.environ["SPARK_HOME"])
!ls -la $SPARK_HOME
!ls -la $SPARK_HOME/python
!ls -la $SPARK_HOME/python/lib

import findspark
findspark.init()

import pyspark
print("PySpark version:", pyspark.__version__)

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--2026-04-18 17:57:42--  https://archive.apache.org/dist/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2

In [4]:
!echo "==== Java JDK: $JAVA_HOME ===="
!/usr/lib/jvm/java-21-openjdk-amd64/bin/java --version
!echo "==== SPARK Binaries: $SPARK_HOME ===="
!ls $SPARK_HOME/bin/
!echo "================================"
!$SPARK_HOME/bin/spark-shell --version
!echo "==== Colab Working Directory ===="
!pwd

==== Java JDK: /usr/lib/jvm/java-21-openjdk-amd64 ====
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk 21.0.10 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)
==== SPARK Binaries: /content/spark-4.0.1-bin-hadoop3 ====
beeline		      pyspark		spark-class.cmd      spark-shell.cmd
beeline.cmd	      pyspark2.cmd	spark-connect-shell  spark-sql
docker-image-tool.sh  pyspark.cmd	sparkR		     spark-sql2.cmd
find-spark-home       run-example	sparkR2.cmd	     spark-sql.cmd
find-spark-home.cmd   run-example.cmd	sparkR.cmd	     spark-submit
load-spark-env.cmd    spark-class	spark-shell	     spark-submit2.cmd
load-s

In [5]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1,"
    "org.apache.spark:spark-avro_2.13:4.0.1 "
    "pyspark-shell"
)

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("group_7_eventhub_ingestion") \
    .getOrCreate()

print("Spark OK", spark.version)

Spark OK 4.0.1


In [7]:
event_hub_namespace = "iesstsabbadbaa-grp-06-10"

orders_eventhub_name = "group_7_orders"
delivery_eventhub_name = "group_7_delivery"

producer_orders_connection_str = "Endpoint=sb://iesstsabbadbaa-grp-06-10.servicebus.windows.net/;SharedAccessKeyName=orders_send;SharedAccessKey=3UJSRhpX6bjpl4R2HjakBgmEbkC9mw4mc+AEhG60Qns=;EntityPath=group_7_orders"
consumer_orders_connection_str = "Endpoint=sb://iesstsabbadbaa-grp-06-10.servicebus.windows.net/;SharedAccessKeyName=orders_listen;SharedAccessKey=MlbHaVdO4YiSjRTdHpLUaUNov39ooEstg+AEhJFQPQA=;EntityPath=group_7_orders"

producer_delivery_connection_str = "Endpoint=sb://iesstsabbadbaa-grp-06-10.servicebus.windows.net/;SharedAccessKeyName=send_delivery;SharedAccessKey=6W0IJkT1iMON2ZEjIGTEj+kaeYHJYKT23+AEhDQjb0k=;EntityPath=group_7_delivery"
consumer_delivery_connection_str = "Endpoint=sb://iesstsabbadbaa-grp-06-10.servicebus.windows.net/;SharedAccessKeyName=listen_delivery;SharedAccessKey=yfAvBw9KehdlJxNQEjmAJiOEuiEJi4ADk+AEhKY8l5Y=;EntityPath=group_7_delivery"

In [8]:
orders_kafka_options = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{consumer_orders_connection_str}";',
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

In [9]:
delivery_kafka_options = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{consumer_delivery_connection_str}";',
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

In [51]:
orders_raw = (
    spark.readStream
    .format("kafka")
    .options(**orders_kafka_options)
    .option("subscribe", orders_eventhub_name)
    .load()
)

In [52]:
delivery_raw = (
    spark.readStream
    .format("kafka")
    .options(**delivery_kafka_options)
    .option("subscribe", delivery_eventhub_name)
    .load()
)

In [53]:
orders_raw.printSchema()
delivery_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [54]:
orders_query = (
    orders_raw
    .writeStream
    .format("console")
    .option("truncate", False)
    .start()
)

In [55]:
orders_query.stop()

In [56]:
delivery_query = (
    delivery_raw
    .writeStream
    .format("console")
    .option("truncate", False)
    .start()
)

In [57]:
delivery_query.stop()

In [17]:
import json
from pyspark.sql.avro.functions import from_avro
from pyspark.sql.functions import col

In [18]:
with open("order_events_v1.avsc", "r") as f:
    order_schema = json.dumps(json.load(f))

In [44]:
orders_parsed = (
    orders_raw
    .select(from_avro(col("value"), order_schema).alias("data"))
    .select("data.*")
)

In [20]:
orders_parsed.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- order_value: double (nullable = true)
 |-- is_promo: boolean (nullable = true)
 |-- estimated_delivery_minutes: integer (nullable = true)
 |-- version: integer (nullable = true)



In [21]:
with open("courier_events_v1.avsc", "r") as f:
    delivery_schema = json.dumps(json.load(f))

In [22]:
delivery_parsed = (
    delivery_raw
    .select(from_avro(col("value"), delivery_schema).alias("data"))
    .select("data.*")
)

In [23]:
delivery_parsed.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- courier_status: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- version: integer (nullable = true)



In [58]:
orders_preview = (
    orders_parsed
    .writeStream
    .format("console")
    .option("truncate", False)
    .start()
)

In [59]:
orders_preview.stop()

In [26]:
delivery_preview = (
    delivery_parsed
    .writeStream
    .format("console")
    .option("truncate", False)
    .start()
)

In [60]:
delivery_preview.stop()

In [31]:
from pyspark.sql.functions import col

orders_silver = (
    orders_parsed
    .withColumn("event_time", col("event_ts"))
    .withColumn("ingest_time", col("ingest_ts"))
    .withWatermark("event_time", "10 minutes")
)

delivery_silver = (
    delivery_parsed
    .withColumn("event_time", col("event_ts"))
    .withColumn("ingest_time", col("ingest_ts"))
    .withWatermark("event_time", "10 minutes")
)

In [61]:
orders_silver.printSchema()
delivery_silver.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- order_value: double (nullable = true)
 |-- is_promo: boolean (nullable = true)
 |-- estimated_delivery_minutes: integer (nullable = true)
 |-- version: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- ingest_time: timestamp (nullable = true)

root
 |-- event_id: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- courier_status: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |--

In [41]:
orders_bronze_query = (
    orders_silver.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", "/content/bronze/orders")
    .option("checkpointLocation", "/content/checkpoints/orders")
    .start()
)

In [50]:
delivery_bronze_query = (
    delivery_silver.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", "/content/bronze/delivery")
    .option("checkpointLocation", "/content/checkpoints/delivery")
    .start()
)

In [35]:
orders_bronze_query.stop()
delivery_bronze_query.stop()

In [62]:
bronze_orders_df = spark.read.parquet("/content/bronze/orders")
bronze_delivery_df = spark.read.parquet("/content/bronze/delivery")

bronze_orders_df.show(5, truncate=False)
bronze_delivery_df.show(5, truncate=False)

+------------------------------------+------------------------------------+----------------+-----------------------+-----------------------+-------------+-------+----------+-----------+--------+--------------------------+-------+-----------------------+-----------------------+
|event_id                            |order_id                            |event_type      |event_ts               |ingest_ts              |restaurant_id|zone_id|courier_id|order_value|is_promo|estimated_delivery_minutes|version|event_time             |ingest_time            |
+------------------------------------+------------------------------------+----------------+-----------------------+-----------------------+-------------+-------+----------+-----------+--------+--------------------------+-------+-----------------------+-----------------------+
|64dce1a6-2081-43d3-a08e-37e1a9b9aa4d|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|prep_started    |2026-04-16 20:19:24.093|2026-04-16 20:36:46.102|rest_20      |east   |NULL